# Experiment 02: Recording-Wise Evaluation

## Objective

The baseline experiment used a random window-level train-test split. This experiment evaluates the trained EEG seizure detection model using a recording-wise split.

The purpose is to investigate whether the model can generalize to EEG recordings that were not used during model training.

### Evaluation Strategy

- EEG windows from the same recording are kept together.
- Recordings are divided into training and testing groups.
- The trained Random Forest model is evaluated on unseen recordings.
- Accuracy, precision, sensitivity (recall), specificity, F1-score, and balanced accuracy are calculated.
- The confusion matrix is generated.
- The results are saved for comparison with the baseline experiment.

### Research Question

> Can the trained model detect seizure-like EEG activity in recordings that were not used during training?

---

**Note:** This experiment is part of a research and learning prototype and is not a clinically validated diagnostic system.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
IMAGES_DIR = PROJECT_ROOT / "images"

FEATURES_PATH = DATA_DIR / "features.csv"
METADATA_PATH = DATA_DIR / "window_metadata.csv"

MODEL_PATH = MODEL_DIR / "random_forest_model.pkl"

RESULTS_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Features Path:", FEATURES_PATH)
print("Metadata Path:", METADATA_PATH)
print("Model Path:", MODEL_PATH)

Project Root: C:\Users\Prajapati_Shivam\EEG-Seizure-Detection
Features Path: C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\features.csv
Metadata Path: C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\window_metadata.csv
Model Path: C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\models\random_forest_model.pkl


In [3]:
features_df = pd.read_csv(FEATURES_PATH)

print("Feature Dataset Loaded Successfully")
print("=" * 60)

print("Dataset Shape:", features_df.shape)

print("\nColumns:")
print(features_df.columns.tolist())

print("\nFirst 5 Rows:")
display(features_df.head())

Feature Dataset Loaded Successfully
Dataset Shape: (13181, 9)

Columns:
['Mean', 'Std', 'Variance', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Label']

First 5 Rows:


,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma,Label
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11,0
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11,0
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11,0
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10,0
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11,0


In [4]:

metadata_df = pd.read_csv(METADATA_PATH)

print("Window Metadata Loaded Successfully")
print("=" * 60)

print("Metadata Shape:", metadata_df.shape)

print("\nMetadata Columns:")
print(metadata_df.columns.tolist())

print("\nFirst 5 Rows:")
display(metadata_df.head())

Window Metadata Loaded Successfully
Metadata Shape: (13181, 4)

Metadata Columns:
['file', 'window_start', 'window_end', 'label']

First 5 Rows:


,file,window_start,window_end,label
0,chb01_01.edf,0.0,4.0,0
1,chb01_01.edf,4.0,8.0,0
2,chb01_01.edf,8.0,12.0,0
3,chb01_01.edf,12.0,16.0,0
4,chb01_01.edf,16.0,20.0,0


In [5]:

print("Feature rows:", len(features_df))
print("Metadata rows:", len(metadata_df))

if len(features_df) == len(metadata_df):
    print("\nSUCCESS: Feature rows and metadata rows are aligned.")
else:
    print("\nERROR: Feature rows and metadata rows are NOT aligned.")

Feature rows: 13181
Metadata rows: 13181

SUCCESS: Feature rows and metadata rows are aligned.


In [6]:
print("Unique EEG Recordings")
print("=" * 60)

recording_summary = (
    metadata_df
    .groupby("file")
    .agg(
        total_windows=("label", "count"),
        seizure_windows=("label", "sum")
    )
    .reset_index()
)

recording_summary["normal_windows"] = (
    recording_summary["total_windows"]
    - recording_summary["seizure_windows"]
)

display(recording_summary)

print("\nTotal Recordings:", len(recording_summary))

Unique EEG Recordings


,file,total_windows,seizure_windows,normal_windows
0,chb01_01.edf,900,0,900
1,chb01_03.edf,900,10,890
2,chb01_04.edf,900,8,892
3,chb01_09.edf,900,0,900
4,chb01_15.edf,900,10,890
5,chb01_18.edf,900,23,877
6,chb01_21.edf,900,24,876
7,chb01_26.edf,581,26,555
8,chb01_30.edf,900,0,900
9,chb01_38.edf,900,0,900



Total Recordings: 15


In [7]:

seizure_recordings = recording_summary[
    recording_summary["seizure_windows"] > 0
]["file"].tolist()

normal_recordings = recording_summary[
    recording_summary["seizure_windows"] == 0
]["file"].tolist()

print("Seizure-containing recordings:")
for file in seizure_recordings:
    print(" -", file)

print("\nNumber of seizure-containing recordings:",
      len(seizure_recordings))

print("\nNormal-only recordings:")
for file in normal_recordings:
    print(" -", file)

print("\nNumber of normal-only recordings:",
      len(normal_recordings))

Seizure-containing recordings:
 - chb01_03.edf
 - chb01_04.edf
 - chb01_15.edf
 - chb01_18.edf
 - chb01_21.edf
 - chb01_26.edf

Number of seizure-containing recordings: 6

Normal-only recordings:
 - chb01_01.edf
 - chb01_09.edf
 - chb01_30.edf
 - chb01_38.edf
 - chb01_39.edf
 - chb01_40.edf
 - chb01_41.edf
 - chb01_42.edf
 - chb01_46.edf

Number of normal-only recordings: 9


In [8]:

# Selected unseen test recordings
test_recordings = [
    "chb01_03.edf",
    "chb01_15.edf",
    "chb01_01.edf",
    "chb01_09.edf",
    "chb01_30.edf"
]

# All recordings in the dataset
all_recordings = metadata_df["file"].unique().tolist()

# Training recordings = all recordings not used for testing
train_recordings = [
    recording
    for recording in all_recordings
    if recording not in test_recordings
]

print("TRAINING RECORDINGS")
print("=" * 60)

for recording in train_recordings:
    print(" -", recording)

print("\nNumber of training recordings:",
      len(train_recordings))

print("\nTESTING RECORDINGS")
print("=" * 60)

for recording in test_recordings:
    print(" -", recording)

print("\nNumber of testing recordings:",
      len(test_recordings))

TRAINING RECORDINGS
 - chb01_04.edf
 - chb01_18.edf
 - chb01_21.edf
 - chb01_26.edf
 - chb01_38.edf
 - chb01_39.edf
 - chb01_40.edf
 - chb01_41.edf
 - chb01_42.edf
 - chb01_46.edf

Number of training recordings: 10

TESTING RECORDINGS
 - chb01_03.edf
 - chb01_15.edf
 - chb01_01.edf
 - chb01_09.edf
 - chb01_30.edf

Number of testing recordings: 5


In [9]:
# ============================================================
# VERIFY RECORDING LEAKAGE
# ============================================================

train_set = set(train_recordings)
test_set = set(test_recordings)

overlap = train_set.intersection(test_set)

print("Training recordings:", len(train_set))
print("Testing recordings:", len(test_set))

print("\nOverlapping recordings:")
print(overlap)

if len(overlap) == 0:
    print("\nSUCCESS: No recording leakage detected.")
else:
    print("\nERROR: Recording leakage detected!")

Training recordings: 10
Testing recordings: 5

Overlapping recordings:
set()

SUCCESS: No recording leakage detected.


In [10]:
# ============================================================
# CREATE RECORDING-WISE TRAINING AND TESTING DATASETS
# ============================================================

# Feature columns used by the Random Forest model
feature_names = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

# Create masks based on recording names
train_mask = metadata_df["file"].isin(train_recordings)
test_mask = metadata_df["file"].isin(test_recordings)

# Create feature matrices
X_train_recording = features_df.loc[
    train_mask,
    feature_names
]

X_test_recording = features_df.loc[
    test_mask,
    feature_names
]

# Create labels
y_train_recording = features_df.loc[
    train_mask,
    "Label"
]

y_test_recording = features_df.loc[
    test_mask,
    "Label"
]

print("Recording-Wise Dataset")
print("=" * 60)

print("\nTraining Features Shape:",
      X_train_recording.shape)

print("Training Labels Shape:",
      y_train_recording.shape)

print("\nTesting Features Shape:",
      X_test_recording.shape)

print("Testing Labels Shape:",
      y_test_recording.shape)

Recording-Wise Dataset

Training Features Shape: (8681, 8)
Training Labels Shape: (8681,)

Testing Features Shape: (4500, 8)
Testing Labels Shape: (4500,)


In [11]:
# ============================================================
# CHECK RECORDING-WISE CLASS DISTRIBUTION
# ============================================================

print("TRAINING CLASS DISTRIBUTION")
print("=" * 60)

print(
    y_train_recording.value_counts()
    .sort_index()
)

print("\nTesting CLASS DISTRIBUTION")
print("=" * 60)

print(
    y_test_recording.value_counts()
    .sort_index()
)

TRAINING CLASS DISTRIBUTION
Label
0    8600
1      81
Name: count, dtype: int64

Testing CLASS DISTRIBUTION
Label
0    4480
1      20
Name: count, dtype: int64


In [12]:
# ============================================================
# LOAD EXISTING RANDOM FOREST MODEL
# ============================================================

model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")
print("=" * 60)

print("Model Type:", type(model).__name__)

print("\nModel Parameters:")
print(model)

Model loaded successfully.
Model Type: RandomForestClassifier

Model Parameters:
RandomForestClassifier(class_weight='balanced', random_state=42)


In [13]:
# ============================================================
# GENERATE PREDICTIONS ON UNSEEN RECORDINGS
# ============================================================

# Predict class labels
y_pred_recording = model.predict(
    X_test_recording
)

# Predict seizure probabilities
y_prob_recording = model.predict_proba(
    X_test_recording
)[:, 1]

print("Predictions generated successfully.")
print("=" * 60)

print("Total Test Samples:",
      len(y_pred_recording))

print("\nPredicted Class Distribution:")
print(
    pd.Series(y_pred_recording)
    .value_counts()
    .sort_index()
)

Predictions generated successfully.
Total Test Samples: 4500

Predicted Class Distribution:
0    4492
1       8
Name: count, dtype: int64


In [14]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_recording = confusion_matrix(
    y_test_recording,
    y_pred_recording
)

print("Recording-Wise Confusion Matrix")
print("=" * 60)

print(cm_recording)

Recording-Wise Confusion Matrix
[[4472    8]
 [  20    0]]


In [15]:
# ============================================================
# CONFUSION MATRIX COMPONENTS
# ============================================================

tn, fp, fn, tp = cm_recording.ravel()

print("True Negatives (TN):", tn)
print("False Positives (FP):", fp)
print("False Negatives (FN):", fn)
print("True Positives (TP):", tp)

True Negatives (TN): 4472
False Positives (FP): 8
False Negatives (FN): 20
True Positives (TP): 0


In [16]:
# ============================================================
# RECORDING-WISE MODEL METRICS
# ============================================================

accuracy_recording = accuracy_score(
    y_test_recording,
    y_pred_recording
)

precision_recording = precision_score(
    y_test_recording,
    y_pred_recording,
    zero_division=0
)

sensitivity_recording = recall_score(
    y_test_recording,
    y_pred_recording,
    zero_division=0
)

specificity_recording = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)

f1_recording = f1_score(
    y_test_recording,
    y_pred_recording,
    zero_division=0
)

balanced_accuracy_recording = balanced_accuracy_score(
    y_test_recording,
    y_pred_recording
)

print("RECORDING-WISE EVALUATION METRICS")
print("=" * 60)

print(f"Accuracy:           {accuracy_recording:.4f}")
print(f"Precision:          {precision_recording:.4f}")
print(f"Sensitivity/Recall: {sensitivity_recording:.4f}")
print(f"Specificity:        {specificity_recording:.4f}")
print(f"F1-Score:           {f1_recording:.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_recording:.4f}")

RECORDING-WISE EVALUATION METRICS
Accuracy:           0.9938
Precision:          0.0000
Sensitivity/Recall: 0.0000
Specificity:        0.9982
F1-Score:           0.0000
Balanced Accuracy:  0.4991


In [17]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

recording_report = classification_report(
    y_test_recording,
    y_pred_recording,
    target_names=["Normal", "Seizure"],
    zero_division=0
)

print("RECORDING-WISE CLASSIFICATION REPORT")
print("=" * 60)

print(recording_report)

RECORDING-WISE CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      4480
     Seizure       0.00      0.00      0.00        20

    accuracy                           0.99      4500
   macro avg       0.50      0.50      0.50      4500
weighted avg       0.99      0.99      0.99      4500

